In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 10_inference_pipeline_from_csv (CORREGIDO)
# MAGIC Carga desde CSV con manejo correcto de tipos

# COMMAND ----------

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from datetime import datetime, timedelta

# Configuración de rutas
CSV_PATH = "/Volumes/olist/olist_csv/olist2/"
MODELS_PATH = "/Volumes/olist/olist_gold/models/"
INFERENCE_PATH = "/Volumes/olist/olist_gold/inference/"

# Configuración de periodo
USAR_PERIODO_AUTOMATICO = True
START_DATE = None
END_DATE = None
CUTOFF_DATE = None

print("="*80)
print("🚀 PIPELINE DE INFERENCIA - DESDE CSV ORIGINALES")
print("="*80)
print(f"\n📂 Fuente: {CSV_PATH}")
print(f"⚙️  Modo: {'AUTOMÁTICO' if USAR_PERIODO_AUTOMATICO else 'MANUAL'}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 0: Verificación de CSV y Carga de Artefactos

# COMMAND ----------

print("🔍 ETAPA 0: VERIFICACIÓN\n" + "="*80 + "\n")

# Listar archivos CSV
print("📂 Archivos CSV disponibles:")
csv_files = dbutils.fs.ls(CSV_PATH)
for file in sorted(csv_files, key=lambda x: x.name):
    size_mb = file.size / (1024 * 1024)
    print(f"   • {file.name:45s} {size_mb:>8.2f} MB")
print()

# Verificar archivos críticos
required_files = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_customers_dataset.csv"
]

missing = [f for f in required_files if not any(x.name == f for x in csv_files)]
if missing:
    raise FileNotFoundError(f"Archivos faltantes: {missing}")

print("✅ Todos los archivos requeridos disponibles\n")

# Cargar artefactos del modelo
print("📦 Cargando artefactos del modelo...")
try:
    metadata = spark.read.format("delta").load(f"{MODELS_PATH}transformation_metadata/").toPandas()
    n_pca_expected = int(metadata['n_pca_components'].iloc[0])
    features_retained_df = spark.read.format("delta").load(f"{MODELS_PATH}features_retained/").toPandas()
    features_retained = features_retained_df.sort_values('order')['feature'].tolist()
    
    print(f"✅ Artefactos cargados:")
    print(f"   • Componentes PCA: {n_pca_expected}")
    print(f"   • Features retenidas: {len(features_retained)}\n")
except Exception as e:
    print(f"❌ Error: {e}\n⚠️  Ejecuta primero: 06b_pca_save_simple")
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 1: Carga de CSV con Tipos Correctos

# COMMAND ----------

print("📥 ETAPA 1: CARGA Y PROCESAMIENTO DE CSV\n" + "="*80 + "\n")

# 1.1 Cargar orders (con conversión explícita de fechas)
print("1️⃣ Cargando olist_orders_dataset.csv...")
orders = spark.read.csv(
    f"{CSV_PATH}olist_orders_dataset.csv",
    header=True,
    inferSchema=False
)

# Convertir fechas explícitamente
orders = orders \
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))

print(f"   ✅ {orders.count():,} órdenes cargadas\n")

# 1.2 Cargar order_items (con conversión explícita de numéricos)
print("2️⃣ Cargando olist_order_items_dataset.csv...")
order_items = spark.read.csv(
    f"{CSV_PATH}olist_order_items_dataset.csv",
    header=True,
    inferSchema=False
)

order_items = order_items \
    .withColumn("order_item_id", F.col("order_item_id").cast("int")) \
    .withColumn("price", F.col("price").cast("double")) \
    .withColumn("freight_value", F.col("freight_value").cast("double"))

print(f"   ✅ {order_items.count():,} items cargados\n")

# 1.3 Cargar payments (con conversión explícita)
print("3️⃣ Cargando olist_order_payments_dataset.csv...")
payments = spark.read.csv(
    f"{CSV_PATH}olist_order_payments_dataset.csv",
    header=True,
    inferSchema=False
)

payments = payments \
    .withColumn("payment_sequential", F.col("payment_sequential").cast("int")) \
    .withColumn("payment_installments", F.col("payment_installments").cast("int")) \
    .withColumn("payment_value", F.col("payment_value").cast("double"))

print(f"   ✅ {payments.count():,} pagos cargados\n")

# 1.4 Cargar reviews (CON try_cast para review_score)
print("4️⃣ Cargando olist_order_reviews_dataset.csv...")
reviews = spark.read.csv(
    f"{CSV_PATH}olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=False
)

reviews = reviews \
    .withColumn("review_creation_date", F.to_timestamp("review_creation_date")) \
    .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp")) \
    .withColumn("review_score", F.expr("try_cast(review_score as int)"))

# Filtrar reviews con score válido
reviews = reviews.filter(F.col("review_score").isNotNull())

print(f"   ✅ {reviews.count():,} reviews válidas cargadas\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 2: Agregar y Combinar Datos

# COMMAND ----------

print("🔧 ETAPA 2: AGREGACIÓN DE DATOS\n" + "="*80 + "\n")

# 2.1 Agregar items por orden
print("📦 Agregando items por orden...")
items_agg = order_items.groupBy("order_id").agg(
    F.count("*").alias("items_count"),
    F.countDistinct("product_id").alias("distinct_products"),
    F.sum("price").alias("sum_price"),
    F.sum("freight_value").alias("sum_freight")
)
print(f"   ✅ {items_agg.count():,} órdenes agregadas\n")

# 2.2 Agregar payments por orden
print("💳 Agregando pagos por orden...")
payments_agg = payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("payment_sum"),
    F.avg("payment_installments").alias("avg_installments"),
    F.countDistinct("payment_type").alias("n_payment_types")
)
print(f"   ✅ {payments_agg.count():,} órdenes agregadas\n")

# 2.3 Agregar reviews por orden
print("⭐ Agregando reviews por orden...")
reviews_agg = reviews.groupBy("order_id").agg(
    F.avg("review_score").alias("avg_review_score"),
    F.min("review_score").alias("min_review_score"),
    F.max("review_score").alias("max_review_score")
)
print(f"   ✅ {reviews_agg.count():,} órdenes agregadas\n")

# 2.4 Combinar todo
print("🔗 Combinando todas las fuentes...")
orders_full = orders \
    .join(items_agg, "order_id", "left") \
    .join(payments_agg, "order_id", "left") \
    .join(reviews_agg, "order_id", "left") \
    .fillna(0)

print(f"   ✅ orders_full: {orders_full.count():,} registros\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 3: Análisis y Selección de Periodo

# COMMAND ----------

print("📅 ETAPA 3: ANÁLISIS DE FECHAS DISPONIBLES\n" + "="*80 + "\n")

# Analizar rango de fechas
date_stats = orders_full.select(
    F.min('order_purchase_timestamp').alias('min_date'),
    F.max('order_purchase_timestamp').alias('max_date'),
    F.count('*').alias('total')
).collect()[0]

min_date = date_stats['min_date']
max_date = date_stats['max_date']
total_orders = date_stats['total']

print(f"📊 DATOS DISPONIBLES:")
print(f"   • Fecha mínima: {min_date}")
print(f"   • Fecha máxima: {max_date}")
print(f"   • Total órdenes: {total_orders:,}\n")

# Configuración de periodo
if USAR_PERIODO_AUTOMATICO:
    max_dt = max_date if isinstance(max_date, datetime) else datetime.strptime(str(max_date), '%Y-%m-%d %H:%M:%S')
    
    PERIODO_DIAS = 60  # ← MODIFICAR AQUÍ (60 días = 2 meses)
    
    start_dt = max_dt - timedelta(days=PERIODO_DIAS)
    
    START_DATE = start_dt.strftime('%Y-%m-%d %H:%M:%S')
    END_DATE = max_dt.strftime('%Y-%m-%d %H:%M:%S')
    CUTOFF_DATE = END_DATE
    
    meses_aprox = PERIODO_DIAS / 30
    print(f"🤖 MODO: AUTOMÁTICO")
    print(f"🗓️  PERIODO SELECCIONADO:")
    print(f"   • Duración: {PERIODO_DIAS} días (~{meses_aprox:.1f} meses)")
    print(f"   • Inicio: {START_DATE}")
    print(f"   • Fin:    {END_DATE}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 4: Filtrado del Periodo

# COMMAND ----------

print("🔍 ETAPA 4: FILTRADO DEL PERIODO\n" + "="*80 + "\n")

orders_production = orders_full.filter(
    (F.col("order_purchase_timestamp") >= F.lit(START_DATE)) &
    (F.col("order_purchase_timestamp") <= F.lit(END_DATE)) &
    (F.col("order_status") != "canceled") &
    (F.col("customer_id").isNotNull())
)

n_orders = orders_production.count()
n_customers = orders_production.select("customer_id").distinct().count()

print(f"📊 RESULTADOS DEL FILTRADO:")
print(f"   • Órdenes: {n_orders:,}")
print(f"   • Clientes: {n_customers:,}\n")

if n_orders == 0:
    raise ValueError("No hay órdenes en el periodo seleccionado")

print(f"✅ Periodo válido con {n_orders:,} órdenes\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 5: Generación de Features (CONSISTENTE CON 05_generate_gold_features)

# COMMAND ----------

print("🎯 ETAPA 5: GENERACIÓN DE FEATURES\n" + "="*80 + "\n")

features = orders_production.groupBy("customer_id").agg(
    # RFM
    F.datediff(F.lit(CUTOFF_DATE), F.max("order_purchase_timestamp")).alias("recency"),
    F.count("order_id").alias("frequency"),
    F.sum("payment_sum").alias("monetary"),
    
    # Ticket promedio/max/min
    F.avg("payment_sum").alias("avg_ticket"),
    F.max("payment_sum").alias("max_ticket"),
    F.min("payment_sum").alias("min_ticket"),
    F.stddev("payment_sum").alias("std_ticket"),
    
    # Items
    F.avg("items_count").alias("avg_items_per_order"),
    F.max("items_count").alias("max_items_per_order"),
    F.sum("items_count").alias("total_items"),
    F.avg("distinct_products").alias("avg_distinct_products"),
    F.sum("distinct_products").alias("total_distinct_products"),
    
    # Precios y flete
    F.avg("sum_price").alias("avg_price"),
    F.sum("sum_price").alias("total_price"),
    F.avg("sum_freight").alias("avg_freight"),
    F.sum("sum_freight").alias("total_freight"),
    
    # Pagos
    F.avg("avg_installments").alias("avg_installments"),
    F.max("avg_installments").alias("max_installments"),
    F.avg("n_payment_types").alias("avg_payment_types"),
    
    # Reviews (sin cast, ya viene como double de reviews_agg)
    F.avg("avg_review_score").alias("avg_review_score"),
    F.min("min_review_score").alias("min_review_score"),
    F.max("max_review_score").alias("max_review_score"),
    F.count(F.when(F.col("avg_review_score").isNotNull(), 1)).alias("orders_with_review"),
    
    # Temporales - primera y última compra
    F.min("order_purchase_timestamp").alias("first_purchase"),
    F.max("order_purchase_timestamp").alias("last_purchase"),
    F.datediff(F.max("order_purchase_timestamp"), F.min("order_purchase_timestamp")).alias("customer_lifetime_days"),
    
    # Entrega
    F.avg(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("avg_delivery_days"),
    F.max(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("max_delivery_days"),
    F.avg(F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")).alias("avg_delay_days"),
    F.count(F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)).alias("delayed_orders"),
    
    # Status
    F.count(F.when(F.col("order_status") == "delivered", 1)).alias("delivered_orders"),
    F.count(F.when(F.col("order_status") == "shipped", 1)).alias("shipped_orders")
)

print(f"✅ Features base generadas: {len(features.columns)} columnas\n")

# COMMAND ----------

# Features temporales (mes/día de primera y última compra)
print("📅 Agregando features temporales...\n")

features = features \
    .withColumn("first_purchase_month", F.month("first_purchase")) \
    .withColumn("first_purchase_day", F.dayofmonth("first_purchase")) \
    .withColumn("first_purchase_dow", F.dayofweek("first_purchase")) \
    .withColumn("last_purchase_month", F.month("last_purchase")) \
    .withColumn("last_purchase_day", F.dayofmonth("last_purchase")) \
    .withColumn("last_purchase_dow", F.dayofweek("last_purchase"))

# Quitar columnas timestamp originales
features = features.drop("first_purchase", "last_purchase")

print(f"✅ Features temporales agregadas\n")

# COMMAND ----------

# Features de interacción (usando try_divide - IGUAL QUE 05_generate_gold_features)
print("🔗 Creando features de interacción...\n")

features = features \
    .withColumn("freight_price_ratio", F.expr("try_divide(total_freight, total_price)")) \
    .withColumn("monetary_per_order", F.expr("try_divide(monetary, frequency)")) \
    .withColumn("items_per_monetary", F.expr("try_divide(total_items, monetary)")) \
    .withColumn("products_per_order", F.expr("try_divide(total_distinct_products, frequency)")) \
    .withColumn("review_score_x_monetary", F.col("avg_review_score") * F.col("monetary")) \
    .withColumn("delayed_ratio", F.expr("try_divide(delayed_orders, frequency)")) \
    .withColumn("delivered_ratio", F.expr("try_divide(delivered_orders, frequency)")) \
    .withColumn("orders_per_day", F.expr("try_divide(frequency, customer_lifetime_days)"))

print(f"✅ Features de interacción creadas\n")

# COMMAND ----------

# Rellenar NaNs con 0
print("🧹 Limpiando NaNs...\n")

features = features.fillna(0)

customer_features_raw = features.toPandas()

print(f"✅ Total features: {len(features.columns)}\n")
print(f"✅ Features generadas: {customer_features_raw.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 6: Selección de Features

# COMMAND ----------

print("✂️  ETAPA 6: SELECCIÓN DE FEATURES\n" + "="*80 + "\n")

customer_ids = customer_features_raw['customer_id'].copy()
feature_cols_raw = [c for c in customer_features_raw.columns if c != 'customer_id']

# Alinear con features retenidas
for col in features_retained:
    if col not in customer_features_raw.columns:
        customer_features_raw[col] = 0

customer_features_selected = customer_features_raw[features_retained].copy()
print(f"✅ Features seleccionadas: {customer_features_selected.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 7: Estandarización

# COMMAND ----------

print("📏 ETAPA 7: ESTANDARIZACIÓN\n" + "="*80 + "\n")

scaler_params_df = spark.read.format("delta").load(f"{MODELS_PATH}scaler_params/").toPandas()
scaler_params_df = scaler_params_df.sort_values('feature_index')

scaler = StandardScaler()
scaler.mean_ = scaler_params_df['mean'].values
scaler.scale_ = scaler_params_df['scale'].values
scaler.var_ = scaler_params_df['var'].values
scaler.n_features_in_ = len(scaler_params_df)

customer_features_scaled = scaler.transform(customer_features_selected)
print(f"✅ Estandarizado: {customer_features_scaled.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 8: PCA

# COMMAND ----------

print("🔬 ETAPA 8: PCA\n" + "="*80 + "\n")

pca_components_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_components/").toPandas()
pca_components_df = pca_components_df.sort_values('component_id')
pca_params_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_params/").toPandas()
pca_params_df = pca_params_df.sort_values('component_id')
pca_mean_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_mean/").toPandas()

pca = PCA(n_components=len(pca_params_df))
component_cols = [c for c in pca_components_df.columns if c != 'component_id']
pca.components_ = pca_components_df[component_cols].values
pca.explained_variance_ = pca_params_df['explained_variance'].values
pca.explained_variance_ratio_ = pca_params_df['explained_variance_ratio'].values
pca.singular_values_ = pca_params_df['singular_values'].values
pca.mean_ = pca_mean_df['pca_mean'].values
pca.n_features_in_ = len(pca.mean_)
pca.n_components_ = len(pca.components_)

X_pca = pca.transform(customer_features_scaled)

pca_cols = [f'pca_{i+1}' for i in range(pca.n_components_)]
customer_features_pca = pd.DataFrame(X_pca, columns=pca_cols)
customer_features_pca['customer_id'] = customer_ids.values

print(f"✅ PCA aplicado: {customer_features_pca.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 9: Validación y Guardado

# COMMAND ----------

print("✅ ETAPA 9: VALIDACIÓN Y GUARDADO\n" + "="*80 + "\n")

assert customer_features_pca.isnull().sum().sum() == 0, "Hay valores NaN en el dataset PCA"
print("✓ Validación OK: sin NaNs\n")

# Generar nombre dinámico basado en fechas
start_str = START_DATE[:10].replace('-', '')
end_str = END_DATE[:10].replace('-', '')
output_path = f"{INFERENCE_PATH}customer_features_pca_{start_str}_{end_str}/"

try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.inference")
except:
    pass

spark.createDataFrame(customer_features_pca).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(output_path)

print(f"✅ Guardado: {output_path}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Resumen Final

# COMMAND ----------

print("\n" + "="*80)
print("✅ PIPELINE COMPLETADO - DESDE CSV")
print("="*80)
print(f"\n📂 Fuente: CSV originales en {CSV_PATH}")
print(f"📅 Periodo: {START_DATE} → {END_DATE}")
print(f"📊 Órdenes: {n_orders:,}")
print(f"📊 Clientes: {n_customers:,}")
print(f"📊 Features: {len(features_retained)} → {pca.n_components_} PCA")
print(f"\n💾 Output: {output_path}")
print(f"\n🎯 Dataset listo para predicción")
print("\n" + "="*80)